In [0]:
# ================================================================
# CELL 1 — LOAD SNOWFLAKE CREDENTIALS SECURELY
# ================================================================

SECRET_SCOPE = "customer360-snowflake"

REQUIRED_SECRET_KEYS = [
    "username",
    "password",
    "account"
]

# ------------------------------------------------
# Load secrets
# ------------------------------------------------

try:

    sf_user = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="username"
    )

    sf_password = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="password"
    )

    sf_account = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="account"
    )

except Exception as e:

    raise RuntimeError(
        "Snowflake credential retrieval failed. "
        "Verify the Databricks secret scope and secret keys."
    ) from e


# ------------------------------------------------
# Validate secrets
# ------------------------------------------------

if not sf_user:
    raise ValueError(
        "Snowflake username secret is empty."
    )

if not sf_password:
    raise ValueError(
        "Snowflake password secret is empty."
    )

if not sf_account:
    raise ValueError(
        "Snowflake account secret is empty."
    )


print(
    "PASS — Snowflake credentials loaded "
    "from Databricks Secrets."
)

PASS — Snowflake credentials loaded from Databricks Secrets.


In [0]:
# ================================================================
# CELL 2 — SNOWFLAKE CONNECTION CONFIGURATION
# ================================================================

SNOWFLAKE_DATABASE = "RETAIL_CUSTOMER360"
SNOWFLAKE_SCHEMA = "ANALYTICS_GOLD"
SNOWFLAKE_WAREHOUSE = "COMPUTE_WH"
SNOWFLAKE_ROLE = "CUSTOMER360_DATABRICKS_ROLE"
SNOWFLAKE_TARGET_TABLE = "CUSTOMER_360"

SNOWFLAKE_HOST = (
    f"{sf_account}.snowflakecomputing.com"
)


# ------------------------------------------------
# WRITE configuration
# Serverless-compatible
# ------------------------------------------------

sf_write_options = {

    "host": SNOWFLAKE_HOST,
    "port": "443",

    "sfUser": sf_user,
    "sfPassword": sf_password,

    "sfDatabase": SNOWFLAKE_DATABASE,
    "sfSchema": SNOWFLAKE_SCHEMA,

    "sfWarehouse": SNOWFLAKE_WAREHOUSE,
    "sfRole": SNOWFLAKE_ROLE
}


# ------------------------------------------------
# READ configuration
# ------------------------------------------------

sf_read_options = {

    "sfURL": SNOWFLAKE_HOST,

    "sfUser": sf_user,
    "sfPassword": sf_password,

    "sfDatabase": SNOWFLAKE_DATABASE,
    "sfSchema": SNOWFLAKE_SCHEMA,

    "sfWarehouse": SNOWFLAKE_WAREHOUSE,
    "sfRole": SNOWFLAKE_ROLE
}


print(
    f"Snowflake host      : {SNOWFLAKE_HOST}"
)

print(
    f"Snowflake database  : {SNOWFLAKE_DATABASE}"
)

print(
    f"Snowflake schema    : {SNOWFLAKE_SCHEMA}"
)

print(
    f"Snowflake warehouse : {SNOWFLAKE_WAREHOUSE}"
)

print(
    f"Snowflake role      : {SNOWFLAKE_ROLE}"
)

print(
    "PASS — Snowflake READ/WRITE "
    "configuration prepared."
)

Snowflake host      : [REDACTED].snowflakecomputing.com
Snowflake database  : RETAIL_CUSTOMER360
Snowflake schema    : ANALYTICS_GOLD
Snowflake warehouse : COMPUTE_WH
Snowflake role      : CUSTOMER360_DATABRICKS_ROLE
PASS — Snowflake READ/WRITE configuration prepared.


In [0]:
# ================================================================
# CELL 3 — SNOWFLAKE CONNECTION TEST
# ================================================================

try:

    connection_test_df = (
        spark.read
        .format("snowflake")
        .options(**sf_read_options)
        .option(
            "query",
            """
            SELECT
                CURRENT_VERSION() AS SNOWFLAKE_VERSION,
                CURRENT_USER() AS SNOWFLAKE_USER,
                CURRENT_ROLE() AS SNOWFLAKE_ROLE,
                CURRENT_DATABASE() AS SNOWFLAKE_DATABASE,
                CURRENT_SCHEMA() AS SNOWFLAKE_SCHEMA,
                CURRENT_WAREHOUSE() AS SNOWFLAKE_WAREHOUSE
            """
        )
        .load()
    )

    connection_result = (
        connection_test_df
        .first()
        .asDict()
    )

except Exception as e:

    raise RuntimeError(
        "Databricks → Snowflake connection failed."
    ) from e


actual_database = connection_result[
    "SNOWFLAKE_DATABASE"
]

actual_schema = connection_result[
    "SNOWFLAKE_SCHEMA"
]

actual_role = connection_result[
    "SNOWFLAKE_ROLE"
]

actual_warehouse = connection_result[
    "SNOWFLAKE_WAREHOUSE"
]


print(
    f"Database  : {actual_database}"
)

print(
    f"Schema    : {actual_schema}"
)

print(
    f"Role      : {actual_role}"
)

print(
    f"Warehouse : {actual_warehouse}"
)

Database  : RETAIL_CUSTOMER360
Schema    : ANALYTICS_GOLD
Role      : CUSTOMER360_DATABRICKS_ROLE
Warehouse : COMPUTE_WH


In [0]:
# ================================================================
# CELL 4 — SNOWFLAKE ENVIRONMENT QUALITY GATE
# ================================================================

environment_errors = {}

if actual_database != SNOWFLAKE_DATABASE:

    environment_errors["database"] = {
        "expected": SNOWFLAKE_DATABASE,
        "actual": actual_database
    }


if actual_schema != SNOWFLAKE_SCHEMA:

    environment_errors["schema"] = {
        "expected": SNOWFLAKE_SCHEMA,
        "actual": actual_schema
    }


if actual_role != SNOWFLAKE_ROLE:

    environment_errors["role"] = {
        "expected": SNOWFLAKE_ROLE,
        "actual": actual_role
    }


if actual_warehouse != SNOWFLAKE_WAREHOUSE:

    environment_errors["warehouse"] = {
        "expected": SNOWFLAKE_WAREHOUSE,
        "actual": actual_warehouse
    }


if environment_errors:

    print(
        "Snowflake environment validation errors:"
    )

    print(environment_errors)

    raise ValueError(
        "Connected to an unexpected Snowflake environment."
    )


print(
    "PASS — Databricks is connected to "
    "the correct Snowflake environment."
)

PASS — Databricks is connected to the correct Snowflake environment.


In [0]:
# ================================================================
# CELL 5 — LOAD VALIDATED GOLD CUSTOMER 360
# ================================================================

GOLD_TABLE = "workspace.gold.customer_360"

try:

    gold_customer_360 = spark.table(
        GOLD_TABLE
    )

except Exception as e:

    raise RuntimeError(
        f"Unable to load Gold table: {GOLD_TABLE}"
    ) from e


source_row_count = (
    gold_customer_360.count()
)

print(
    f"Gold Customer 360 source rows : "
    f"{source_row_count:,}"
)

Gold Customer 360 source rows : 93,358


In [0]:
# ================================================================
# CELL 6 — GOLD PRE-PUBLICATION SAFETY GATE
# ================================================================

from pyspark.sql import functions as F


# ------------------------------------------------
# 1. EMPTY DATASET CHECK
# ------------------------------------------------

if source_row_count == 0:

    raise ValueError(
        "Gold Customer 360 contains zero rows. "
        "Snowflake publication aborted."
    )


# ------------------------------------------------
# 2. REQUIRED COLUMN CHECK
# ------------------------------------------------

required_columns = [

    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix",
    "latitude",
    "longitude",

    "recency_days",
    "frequency",
    "monetary",

    "avg_review_score",

    "r_score",
    "f_score",
    "m_score",

    "rfm_score",
    "rfm_segment",

    "gold_load_timestamp"
]


missing_columns = [
    column
    for column in required_columns
    if column not in gold_customer_360.columns
]


if missing_columns:

    raise ValueError(
        "Gold schema validation failed. "
        f"Missing columns: {missing_columns}"
    )


# ------------------------------------------------
# 3. NULL CUSTOMER KEY CHECK
# ------------------------------------------------

null_customer_count = (
    gold_customer_360
    .filter(
        F.col("customer_unique_id").isNull()
        |
        (
            F.trim(
                F.col("customer_unique_id")
            ) == ""
        )
    )
    .count()
)


if null_customer_count != 0:

    raise ValueError(
        "Gold Customer 360 contains "
        f"{null_customer_count:,} NULL/blank "
        "customer_unique_id values."
    )


# ------------------------------------------------
# 4. DUPLICATE CUSTOMER KEY CHECK
# ------------------------------------------------

duplicate_customer_count = (
    gold_customer_360
    .groupBy("customer_unique_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


if duplicate_customer_count != 0:

    raise ValueError(
        "Gold Customer 360 contains "
        f"{duplicate_customer_count:,} "
        "duplicate customer keys."
    )


# ------------------------------------------------
# 5. REQUIRED METRIC NULL CHECK
# ------------------------------------------------

required_metrics = [
    "recency_days",
    "frequency",
    "monetary",
    "r_score",
    "f_score",
    "m_score",
    "rfm_score",
    "rfm_segment",
    "gold_load_timestamp"
]


metric_null_errors = {}

for column_name in required_metrics:

    invalid_count = (
        gold_customer_360
        .filter(
            F.col(column_name).isNull()
        )
        .count()
    )

    if invalid_count != 0:

        metric_null_errors[
            column_name
        ] = invalid_count


if metric_null_errors:

    raise ValueError(
        "Gold required-field validation failed: "
        f"{metric_null_errors}"
    )


print(
    "PASS — Gold Customer 360 passed "
    "pre-publication safety gates."
)

PASS — Gold Customer 360 passed pre-publication safety gates.


In [0]:
# ================================================================
# CELL 7 — GOLD ANALYTICAL VALUE SAFETY GATE
# ================================================================


# ------------------------------------------------
# Recency
# ------------------------------------------------

invalid_recency_count = (
    gold_customer_360
    .filter(
        F.col("recency_days") < 0
    )
    .count()
)


# ------------------------------------------------
# Frequency
# ------------------------------------------------

invalid_frequency_count = (
    gold_customer_360
    .filter(
        F.col("frequency") <= 0
    )
    .count()
)


# ------------------------------------------------
# Monetary
# ------------------------------------------------

invalid_monetary_count = (
    gold_customer_360
    .filter(
        F.col("monetary") < 0
    )
    .count()
)


# ------------------------------------------------
# Review score
# ------------------------------------------------

invalid_review_count = (
    gold_customer_360
    .filter(
        (F.col("avg_review_score") < 1)
        |
        (F.col("avg_review_score") > 5)
    )
    .count()
)


# ------------------------------------------------
# RFM component scores
# ------------------------------------------------

invalid_rfm_component_count = (
    gold_customer_360
    .filter(
        (F.col("r_score") < 1)
        |
        (F.col("r_score") > 5)
        |
        (F.col("f_score") < 1)
        |
        (F.col("f_score") > 5)
        |
        (F.col("m_score") < 1)
        |
        (F.col("m_score") > 5)
    )
    .count()
)


print(
    f"Invalid recency records       : "
    f"{invalid_recency_count:,}"
)

print(
    f"Invalid frequency records     : "
    f"{invalid_frequency_count:,}"
)

print(
    f"Invalid monetary records      : "
    f"{invalid_monetary_count:,}"
)

print(
    f"Invalid review score records  : "
    f"{invalid_review_count:,}"
)

print(
    f"Invalid RFM component records : "
    f"{invalid_rfm_component_count:,}"
)


if invalid_recency_count != 0:
    raise ValueError(
        "Invalid negative recency values found."
    )


if invalid_frequency_count != 0:
    raise ValueError(
        "Invalid frequency values found."
    )


if invalid_monetary_count != 0:
    raise ValueError(
        "Negative monetary values found."
    )


if invalid_review_count != 0:
    raise ValueError(
        "Invalid review scores found."
    )


if invalid_rfm_component_count != 0:
    raise ValueError(
        "Invalid RFM component scores found."
    )


print(
    "PASS — Gold analytical values are valid."
)

Invalid recency records       : 0
Invalid frequency records     : 0
Invalid monetary records      : 0
Invalid review score records  : 0
Invalid RFM component records : 0
PASS — Gold analytical values are valid.


In [0]:
# ================================================================
# CELL 8 — PUBLISH GOLD CUSTOMER 360 TO SNOWFLAKE
# ================================================================

try:

    (
        gold_customer_360
        .write
        .format("snowflake")
        .options(**sf_write_options)
        .option(
            "dbtable",
            SNOWFLAKE_TARGET_TABLE
        )
        .mode("overwrite")
        .save()
    )

except Exception as e:

    raise RuntimeError(
        "Gold Customer 360 publication to Snowflake FAILED. "
        "Downstream verification will not run."
    ) from e


print("=" * 72)

print(
    "GOLD CUSTOMER 360 PUBLICATION"
)

print("=" * 72)

print(
    f"Source : {GOLD_TABLE}"
)

print(
    "Target : "
    f"{SNOWFLAKE_DATABASE}."
    f"{SNOWFLAKE_SCHEMA}."
    f"{SNOWFLAKE_TARGET_TABLE}"
)

print(
    f"Rows published : "
    f"{source_row_count:,}"
)

print(
    "PASS — Gold Customer 360 published "
    "successfully to Snowflake."
)

print("=" * 72)

GOLD CUSTOMER 360 PUBLICATION
Source : workspace.gold.customer_360
Target : RETAIL_CUSTOMER360.ANALYTICS_GOLD.CUSTOMER_360
Rows published : 93,358
PASS — Gold Customer 360 published successfully to Snowflake.


In [0]:
# ================================================================
# CELL 9 — SNOWFLAKE ROW COUNT RECONCILIATION
# ================================================================

try:

    snowflake_count_df = (
        spark.read
        .format("snowflake")
        .options(**sf_read_options)
        .option(
            "query",
            f"""
            SELECT COUNT(*) AS ROW_COUNT
            FROM {SNOWFLAKE_DATABASE}.
                 {SNOWFLAKE_SCHEMA}.
                 {SNOWFLAKE_TARGET_TABLE}
            """
        )
        .load()
    )

    snowflake_gold_count = int(
        snowflake_count_df
        .first()["ROW_COUNT"]
    )

except Exception as e:

    raise RuntimeError(
        "Unable to read Snowflake target "
        "for row-count reconciliation."
    ) from e


print(
    f"Databricks Gold rows : "
    f"{source_row_count:,}"
)

print(
    f"Snowflake Gold rows  : "
    f"{snowflake_gold_count:,}"
)


if source_row_count != snowflake_gold_count:

    raise ValueError(
        "Gold row-count reconciliation FAILED. "
        f"Databricks={source_row_count:,}, "
        f"Snowflake={snowflake_gold_count:,}"
    )


print(
    "PASS — Databricks and Snowflake "
    "Gold row counts match exactly."
)

Databricks Gold rows : 93,358
Snowflake Gold rows  : 93,358
PASS — Databricks and Snowflake Gold row counts match exactly.


In [0]:
# ================================================================
# CELL 10 — TWO-WAY CUSTOMER POPULATION RECONCILIATION
# ================================================================

# ------------------------------------------------
# Databricks keys
# ------------------------------------------------

databricks_keys_df = (
    gold_customer_360
    .select("customer_unique_id")
    .distinct()
)


# ------------------------------------------------
# Snowflake keys
# ------------------------------------------------

snowflake_keys_df = (
    spark.read
    .format("snowflake")
    .options(**sf_read_options)
    .option(
        "dbtable",
        SNOWFLAKE_TARGET_TABLE
    )
    .load()
    .select("customer_unique_id")
    .distinct()
)


databricks_key_count = (
    databricks_keys_df.count()
)

snowflake_key_count = (
    snowflake_keys_df.count()
)


# ------------------------------------------------
# Databricks → Snowflake
# ------------------------------------------------

missing_in_snowflake_count = (
    databricks_keys_df
    .join(
        snowflake_keys_df,
        on="customer_unique_id",
        how="left_anti"
    )
    .count()
)


# ------------------------------------------------
# Snowflake → Databricks
# ------------------------------------------------

missing_in_databricks_count = (
    snowflake_keys_df
    .join(
        databricks_keys_df,
        on="customer_unique_id",
        how="left_anti"
    )
    .count()
)


print(
    f"Databricks distinct customers       : "
    f"{databricks_key_count:,}"
)

print(
    f"Snowflake distinct customers        : "
    f"{snowflake_key_count:,}"
)

print(
    f"Databricks keys missing in Snowflake: "
    f"{missing_in_snowflake_count:,}"
)

print(
    f"Snowflake keys missing in Databricks: "
    f"{missing_in_databricks_count:,}"
)


if missing_in_snowflake_count != 0:

    raise ValueError(
        "Databricks contains customers "
        "missing from Snowflake."
    )


if missing_in_databricks_count != 0:

    raise ValueError(
        "Snowflake contains unexpected "
        "customers not present in Databricks."
    )


if databricks_key_count != snowflake_key_count:

    raise ValueError(
        "Distinct customer counts do not match."
    )


print(
    "PASS — Databricks and Snowflake contain "
    "the exact same customer population."
)

Databricks distinct customers       : 93,358
Snowflake distinct customers        : 93,358
Databricks keys missing in Snowflake: 0
Snowflake keys missing in Databricks: 0
PASS — Databricks and Snowflake contain the exact same customer population.


In [0]:
# ================================================================
# CELL 11 — FINAL PIPELINE SUCCESS GATE
# ================================================================

print("=" * 72)
print("DATABRICKS → SNOWFLAKE GOLD PIPELINE")
print("=" * 72)

print(
    f"Gold source rows       : {source_row_count:,}"
)

print(
    f"Snowflake target rows  : {snowflake_gold_count:,}"
)

print(
    f"Customer population    : "
    f"{databricks_key_count:,}"
)

print(
    f"Missing in Snowflake   : "
    f"{missing_in_snowflake_count:,}"
)

print(
    f"Unexpected in Snowflake: "
    f"{missing_in_databricks_count:,}"
)

print("-" * 72)

print(
    "STATUS: PASS — GOLD CUSTOMER 360 "
    "SUCCESSFULLY PUBLISHED AND VERIFIED."
)

print("=" * 72)

DATABRICKS → SNOWFLAKE GOLD PIPELINE
Gold source rows       : 93,358
Snowflake target rows  : 93,358
Customer population    : 93,358
Missing in Snowflake   : 0
Unexpected in Snowflake: 0
------------------------------------------------------------------------
STATUS: PASS — GOLD CUSTOMER 360 SUCCESSFULLY PUBLISHED AND VERIFIED.
